# Day 8: Data Preprocessing and Cleaning

## Introduction

Data preprocessing prepares raw data for reliable analysis and machine learning. Real-world datasets may contain missing values, duplicate records, inconsistent values, or columns stored in unsuitable data types. If these issues are not handled properly, they can affect calculations, produce misleading insights, and reduce the reliability of a machine learning model.

This task focuses on understanding the importance of preprocessing and applying basic data-cleaning techniques using Python and Pandas. A student dataset will be examined to identify data-quality issues, handle missing values, remove duplicate records, standardize column names and values, and correct data types. The cleaned dataset will then be saved as a separate CSV file for future use.

### Learning Objectives
- Understand why data preprocessing is necessary before analysis or model training.
- Identify and handle missing values and duplicate records.
- Improve consistency in column names and categorical values.
- Convert columns to appropriate data types.
- Export and verify the cleaned dataset.

In [6]:
import pandas as pd

df = pd.read_csv("student-data-v2.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (33, 10)

Column names:
['student_id', 'name', 'department', 'year', 'age', 'attendance', 'python_marks', 'ml_marks', 'projects_completed', 'join_date']

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   student_id          33 non-null     str    
 1   name                33 non-null     str    
 2   department          33 non-null     str    
 3   year                33 non-null     int64  
 4   age                 31 non-null     float64
 5   attendance          30 non-null     float64
 6   python_marks        31 non-null     float64
 7   ml_marks            31 non-null     float64
 8   projects_completed  33 non-null     int64  
 9   join_date           33 non-null     str    
dtypes: float64(4), int64(2), str(4)
memory usage: 2.7 KB

Missing values:
student_id            0
name                  0
departm

## 1. Removing Duplicate Records

Duplicate records can cause the same student's information to be counted more than once, leading to inaccurate analysis. Pandas provides `duplicated()` to identify duplicates and `drop_duplicates()` to remove them while retaining the first occurrence.

In [7]:
# Identify duplicate records
duplicates = df[df.duplicated()]

print("Number of duplicate records:", df.duplicated().sum())
print("\nDuplicate records:")
display(duplicates)

# Remove duplicate records
df = df.drop_duplicates()

print("\nDataset shape after removing duplicates:", df.shape)
print("Remaining duplicate records:", df.duplicated().sum())

# Save the dataset after duplicate removal
df.to_csv("student_data_deduplicated.csv", index=False)

print("\nDataset saved as student_data_deduplicated.csv")

Number of duplicate records: 3

Duplicate records:


,student_id,name,department,year,age,attendance,python_marks,ml_marks,projects_completed,join_date
30,S004,Vihaan,ECE,3,21.0,NaN,72.0,68.0,4,10-07-2023
31,S006,Sai,aiml,4,18.0,67.1,63.0,93.0,4,16-07-2023
32,S013,Isha,Cse,4,20.0,92.9,54.0,86.0,4,06-08-2023



Dataset shape after removing duplicates: (30, 10)
Remaining duplicate records: 0

Dataset saved as student_data_deduplicated.csv


## 2. Handling Missing Values

Missing values can affect data analysis and machine learning results, so they need to be handled appropriately.

* **Age:** Use K-Nearest Neighbors (KNN) imputation to estimate missing values based on similar records.
* **Attendance and Marks:** Use median imputation to fill missing values. The median is less affected by extreme values than the mean.
* **Join Date:** No imputation is required because the column contains no missing values.

After imputation, verify the dataset using `df.isnull().sum()` to check for any remaining missing values.


In [8]:
from sklearn.impute import KNNImputer

# 1. Use median imputation for attendance and marks
median_columns = ["attendance", "python_marks", "ml_marks"]

for column in median_columns:
    df[column] = df[column].fillna(df[column].median())

# 2. Use KNN imputation for age
knn_columns = ["age", "attendance", "python_marks", "ml_marks"]

knn_imputer = KNNImputer(n_neighbors=3)
df[knn_columns] = knn_imputer.fit_transform(df[knn_columns])

# 3. Check remaining missing values
print("Missing values after imputation:")
print(df.isnull().sum())

Missing values after imputation:
student_id            0
name                  0
department            0
year                  0
age                   0
attendance            0
python_marks          0
ml_marks              0
projects_completed    0
join_date             0
dtype: int64


## 3. Standardizing Column Names and Values

Consistent column names and categorical values make datasets easier to understand and process. Descriptive column names reduce ambiguity, while standardizing text removes unnecessary spaces and inconsistent capitalization.

In [9]:
# Rename columns for clarity
df = df.rename(columns={
    "year": "study_year",
    "ml_marks": "machine_learning_marks"
})

# Standardize department values
df["department"] = df["department"].str.strip().str.upper()

print("Updated column names:")
print(df.columns.tolist())

print("\nStandardized department values:")
print(sorted(df["department"].unique()))

print("\nDepartment counts:")
print(df["department"].value_counts())

Updated column names:
['student_id', 'name', 'department', 'study_year', 'age', 'attendance', 'python_marks', 'machine_learning_marks', 'projects_completed', 'join_date']

Standardized department values:
['AIML', 'CSE', 'ECE', 'IT', 'MECH']

Department counts:
department
AIML    10
CSE      8
MECH     5
ECE      4
IT       3
Name: count, dtype: int64


## 4. Correcting Data Types

Data types determine the kind of values a column can store. Imputation methods such as `KNNImputer` can return numerical columns as floating-point values, even when the original data contains whole numbers. Therefore, columns intended to store whole-number values may need to be rounded and converted back to integers.

The `age`, `attendance`, `python_marks`, and `ml_marks` columns are rounded to whole numbers and converted to integer data types. The `join_date` column is converted from strings to datetime values using `pd.to_datetime()` so that dates can be handled correctly.


In [10]:
# Convert whole-number columns back to integers
integer_columns = ["age", "attendance", "python_marks", "machine_learning_marks"]

for column in integer_columns:
    df[column] = df[column].round().astype(int)

# Convert join_date to datetime
df["join_date"] = pd.to_datetime(
    df["join_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

print(df.dtypes)

student_id                           str
name                                 str
department                           str
study_year                         int64
age                                int64
attendance                         int64
python_marks                       int64
machine_learning_marks             int64
projects_completed                 int64
join_date                 datetime64[us]
dtype: object


## 5. Saving and Verifying the Cleaned Dataset

The cleaned dataset is saved separately so the original file remains unchanged. Final checks confirm the dataset dimensions, duplicate count, missing values, and data types.

In [11]:
# Save the final cleaned dataset
df.to_csv("cleaned_student_performance.csv", index=False)

print("Final dataset saved as cleaned_student_performance.csv")
print("\nFinal dataset shape:", df.shape)
print("Remaining duplicate records:", df.duplicated().sum())

print("\nMissing values:")
print(df.isnull().sum())

print("\nFinal data types:")
print(df.dtypes)

Final dataset saved as cleaned_student_performance.csv

Final dataset shape: (30, 10)
Remaining duplicate records: 0

Missing values:
student_id                0
name                      0
department                0
study_year                0
age                       0
attendance                0
python_marks              0
machine_learning_marks    0
projects_completed        0
join_date                 0
dtype: int64

Final data types:
student_id                           str
name                                 str
department                           str
study_year                         int64
age                                int64
attendance                         int64
python_marks                       int64
machine_learning_marks             int64
projects_completed                 int64
join_date                 datetime64[us]
dtype: object


## Conclusion

The student dataset was preprocessed by removing duplicate records, handling missing numerical values using column medians, standardizing column names and department values, and converting joining dates to datetime format. The deduplicated and final datasets were saved separately, preserving the original file for reference. Final checks confirmed 30 records, 10 columns, no remaining duplicate records, and no missing values.